<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c39-header.png" alt="Nextia Learning · Entity Resolution and Record Matching: Is This the Same Thing?" width="100%">

# Solution: Cluster measures

**[Cluster measures](https://learning.nextia-ai.com/courses/record-matching/m07/cluster-measures/)** · Entity Resolution and Record Matching: Is This the Same Thing? · Module 7, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure entities with B-cubed and cluster-pair precision and recall, and find the wrong pairs that hide in large clusters.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 6 MB). |
| **You should know** | Safe clustering, from [Safer clustering](https://learning.nextia-ai.com/courses/record-matching/m07/safer-clustering/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (rapidfuzz 3.14.6, jellyfish 1.3.0, datasketch 2.0.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/record-matching/m07/cluster-measures/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/M07-L04-cluster-measures/cluster-measures-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 8 MB) into a folder `larkfield-match` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-match` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Every customer record and every product listing is **synthetic**: our code made them from invented people and products, and then added realistic errors on purpose (typing errors, nicknames, old addresses, shared family emails). Because the code knows the true person behind each record, you can always check an answer. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C39/data/dataset-card.md) explains the error model.

In [ ]:
# Setup: download the files into larkfield-match/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C39/"
FILES = {
    "data/sources/shop_accounts.csv": "f59f3a9619c35b572650e10663a0ba5f316202d173850689ec9242706fb4174e",
    "data/sources/loyalty_members.csv": "629696f38469898be48cb664e5d69f9c6d438e8622899bbb281bd520bf44af32",
    "data/sources/garden_centres.csv": "2878063d24a38d841f93bb14e07b2d46619fcc4a3909e92ce0238116434d85b2",
    "data/truth.csv": "d951378129bdf30b649250e6d2127787e5630d66aa7cfe5793b1a110e37a06d3",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-match").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-match and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-match.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd` and matplotlib. It defines a few small helpers:

- `source(name)` reads one customer source as it is (`"shop"`, `"loyalty"` or `"garden"`); `records()` reads all three in shared columns (`record_id, source, name, email, phone, address, postcode, date_of_birth`), with the values exactly as the sources wrote them.
- `truth()` says which true customer (`entity_id`) each record belongs to, and its split (`train`, `validation` or `test`).
- `pair_measures(pairs)` measures a table of pairs (`record_id_l`, `record_id_r`) that a matcher calls "the same": pair precision, recall and F1 on the test split, as in the project's `match eval`.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, pair measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

FILES_OF = {"shop": "shop_accounts.csv", "loyalty": "loyalty_members.csv", "garden": "garden_centres.csv"}

def source(name):
    """One customer source as it is: its own columns, every value as text."""
    return pd.read_csv(f"data/sources/{FILES_OF[name]}", dtype=str, keep_default_na=False)

def records():
    """All customer records in shared columns, values as the sources wrote them."""
    s, l, g = source("shop"), source("loyalty"), source("garden")
    parts = [
        pd.DataFrame({"record_id": s.record_id, "source": "shop", "name": s.first_name + " " + s.last_name,
                      "email": s.email, "phone": s.phone, "address": s.address + ", " + s.city + " " + s.province,
                      "postcode": s.postcode, "date_of_birth": s.date_of_birth}),
        pd.DataFrame({"record_id": l.record_id, "source": "loyalty", "name": l.full_name, "email": l.email,
                      "phone": l.phone, "address": "", "postcode": l.postcode, "date_of_birth": l.date_of_birth}),
        pd.DataFrame({"record_id": g.record_id, "source": "garden", "name": g.name, "email": g.email,
                      "phone": g.phone, "address": g.address, "postcode": "", "date_of_birth": g.date_of_birth}),
    ]
    return pd.concat(parts, ignore_index=True)

def truth():
    """record_id -> entity_id (the true customer) and split."""
    return pd.read_csv("data/truth.csv", dtype=str)

def ordered(pairs):
    """The same pairs with the smaller record ID first, one row per pair."""
    lo = pairs[["record_id_l", "record_id_r"]].min(axis=1)
    hi = pairs[["record_id_l", "record_id_r"]].max(axis=1)
    out = pd.DataFrame({"record_id_l": lo, "record_id_r": hi})
    return out[out.record_id_l != out.record_id_r].drop_duplicates()

def true_pairs():
    """Every pair of records of the same true customer, with its split."""
    t = truth()
    m = t.merge(t, on="entity_id", suffixes=("_l", "_r"))
    m = m[m.record_id_l < m.record_id_r]
    return m[["record_id_l", "record_id_r", "split_l"]].rename(columns={"split_l": "split"})

RANK = {"train": 0, "validation": 1, "test": 2}

def pair_measures(pairs, split="test"):
    """Pair precision, recall and F1 of the pairs a matcher calls the same (a pair is in the test split when one
    of its records belongs to a test customer)."""
    p = ordered(pairs)
    t = truth().set_index("record_id")
    rank = t.split.map(RANK)
    p = p.assign(split=np.maximum(p.record_id_l.map(rank), p.record_id_r.map(rank)).map({v: k for k, v in RANK.items()}),
                 same=(p.record_id_l.map(t.entity_id) == p.record_id_r.map(t.entity_id)))
    tp_all = true_pairs()
    p, tp_all = p[p.split == split], tp_all[tp_all.split == split]
    tp = int(p.same.sum())
    precision = tp / len(p) if len(p) else 0.0
    recall = tp / len(tp_all) if len(tp_all) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3),
            "pairs": len(p), "right": tp, "true_pairs": len(tp_all)}

def same_value_pairs(df, col):
    """Every pair of records that share a non-empty value of col."""
    d = df[df[col].notna() & (df[col] != "")][["record_id", col]]
    sizes = d[col].value_counts()
    n_pairs = int((sizes * (sizes - 1) // 2).sum())
    if n_pairs > 20_000_000:   # protect the computer's memory from a key that is too wide
        raise ValueError(f"This key makes {n_pairs:,} pairs: too many to hold in memory. Use a narrower key.")
    m = d.merge(d, on=col, suffixes=("_l", "_r"))
    return m[m.record_id_l < m.record_id_r][["record_id_l", "record_id_r"]].drop_duplicates()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## Setup: the clean records of Module 2

The next cell cleans every record as Module 2 did: names without accents and titles, split into first name and
surname; emails in lower case; phones as 10 digits; postcodes as 6 characters; dates as YYYY-MM-DD. Run it. It
prints the number of records.

In [ ]:
# The clean records (Module 2): one row per record with first, last, email, phone, postcode, dob.
import re, unicodedata

def strip_accents(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def norm_text(s):
    s = strip_accents(str(s or "")).lower().replace("'", "")
    return " ".join(w for w in re.sub(r"[^\w\s]", " ", s).split() if w not in {"mr", "mrs", "ms", "dr", "m", "mme"})

def norm_phone(s):
    d = re.sub(r"\D", "", re.split(r"(?i)\s*ext", str(s or ""))[0])
    d = d[1:] if len(d) == 11 and d.startswith("1") else d
    return d if len(d) == 10 else ""

def norm_postcode(s):
    p = re.sub(r"[\s-]", "", str(s or "")).upper()
    return p if re.fullmatch(r"[A-Z]\d[A-Z]\d[A-Z]\d", p) else ""

def norm_date(s):
    s = str(s or "")
    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})/(\d{4})", s)
    return f"{m[3]}-{int(m[2]):02d}-{int(m[1]):02d}" if m else (s if re.fullmatch(r"\d{4}-\d\d-\d\d", s) else "")

def split_name(s):
    s = str(s or "")
    if "," in s:
        last, first = s.split(",", 1)
    else:
        words = s.split()
        first, last = " ".join(words[:-1]), (words[-1] if words else "")
    return norm_text(first), norm_text(last)

def postcode_of(postcode, address):
    """The postcode column, or the postcode at the end of a garden-centre address."""
    found = re.findall(r"[A-Za-z]\d[A-Za-z][ -]?\d[A-Za-z]\d", address)
    return norm_postcode(postcode) or (norm_postcode(found[-1]) if found else "")

recs = records()
names = recs.name.map(split_name)
clean = pd.DataFrame({
    "record_id": recs.record_id, "source": recs.source,
    "first": [f for f, _ in names], "last": [l.replace(" ", "") for _, l in names],
    "email": recs.email.str.strip().str.lower(), "phone": recs.phone.map(norm_phone),
    "postcode": [postcode_of(p, a) for p, a in zip(recs.postcode, recs.address)],
    "dob": recs.date_of_birth.map(norm_date),
})
print(f"{len(clean):,} clean records")

## Setup: the matched pairs

The next cell makes the pairs that Module 2's four rules call the same: same email; same phone and surname; same name and date of birth; same name and postcode. Each pair gets a **score**: the number of fields that agree, from 1 to 6. A higher score is a stronger link. Run it. It prints the pairs and their pair measures.

In [ ]:
# The pairs of the four rules (Module 2), with a score: how many of six fields agree.
def key_pairs(cols):
    keep = (clean[cols] != "").all(axis=1)
    d = clean[keep].assign(key=clean[keep][cols].agg("|".join, axis=1))
    return same_value_pairs(d, "key")

rules = pd.concat([key_pairs(["email"]), key_pairs(["phone", "last"]), key_pairs(["first", "last", "dob"]),
                   key_pairs(["first", "last", "postcode"])]).drop_duplicates().reset_index(drop=True)
c = clean.set_index("record_id")
FIELDS = ["first", "last", "email", "phone", "postcode", "dob"]
score = 0
for f in FIELDS:
    a, b = rules.record_id_l.map(c[f]), rules.record_id_r.map(c[f])
    score = score + ((a == b) & (a != "")).astype(int)
rules["score"] = score
print(f"{len(rules):,} pairs called the same;", pair_measures(rules))

## Setup: the clusters of lessons 1 and 2

The next cell joins the pairs into clusters with connected components (lesson 1), then makes them safe (lesson 2): while a cluster holds two different dates of birth or more than 6 records, it cuts the weakest link. Run it. It prints both results.

In [ ]:
# Connected components (lesson 1): every record joined by a chain of pairs ends in one cluster.
def components(pairs, ids):
    parent = {r: r for r in ids}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(pairs.record_id_l, pairs.record_id_r):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[max(ra, rb)] = min(ra, rb)
    return pd.Series({r: find(r) for r in ids}, name="cluster")

ids = clean.record_id.tolist()
cc = components(rules, ids)
sizes = cc.value_counts()
print(f"connected components: {cc.nunique():,} entities (true: 37,500); largest clusters:", sizes.head(5).tolist())

def dob_conflict(dobs):
    """True when the non-empty dates hold two different dates of birth. A day and month swap is the same date."""
    seen = set()
    for d in dobs:
        if not d:
            continue
        swapped = d[:4] + "-" + d[8:10] + "-" + d[5:7]
        if seen and d not in seen and swapped not in seen:
            return True
        seen.add(d)
    return False

# Safe clustering (lesson 2): while a cluster fails a check, cut its weakest link.
MAX_SIZE = 6
dob_of = dict(zip(clean.record_id, clean.dob))

def problems(members):
    why = []
    if len(members) > MAX_SIZE:
        why.append("too many records")
    if dob_conflict([dob_of[r] for r in members]):
        why.append("two dates of birth")
    return why

def pieces(nodes, links):
    """The connected parts of one cluster, after some links were cut."""
    nb = {n: set() for n in nodes}
    for a, b in links:
        nb[a].add(b); nb[b].add(a)
    left, out = set(nodes), []
    while left:
        stack = [left.pop()]; part = set(stack)
        while stack:
            for m in nb[stack.pop()]:
                if m in left:
                    left.remove(m); part.add(m); stack.append(m)
        out.append(part)
    return out

def safe_clusters(pairs, clusters, cuts):
    assign = clusters.to_dict()
    groups = clusters.groupby(clusters).groups
    links = pairs.assign(c=pairs.record_id_l.map(clusters))
    bad = [cid for cid, members in groups.items() if len(members) > 1 and problems(list(members))]
    by_cluster = dict(list(links[links.c.isin(bad)].groupby("c")))
    for cid in bad:
        e = by_cluster[cid]
        weight = {(a, b): s for a, b, s in zip(e.record_id_l, e.record_id_r, e.score)}
        stack = [set(groups[cid])]
        while stack:
            part = stack.pop()
            why = problems(part)
            if len(part) == 1 or not why:
                for r in part:
                    assign[r] = min(part)
                continue
            inside = {k: w for k, w in weight.items() if k[0] in part and k[1] in part}
            weakest = min(inside, key=lambda k: (inside[k], k))
            del weight[weakest]
            cuts.append((*weakest, inside[weakest], ", ".join(why)))
            stack.extend(pieces(part, [k for k in weight if k[0] in part and k[1] in part]))
    return pd.Series(assign, name="cluster")

cuts = []
safe = safe_clusters(rules, cc, cuts)
print(f"safe clustering: {safe.nunique():,} entities, largest clusters:", safe.value_counts().head(5).tolist(),
      f"; {len(cuts):,} links cut")

## 1. A tiny example

Six records of two people, Ann and Ben. One bad link joins all six into one cluster. Run the next cell.

In [ ]:
# Six records of two people: a1, a2, a3 are Ann; b1, b2, b3 are Ben. One bad link (a3-b1) joins them.
tiny_truth = pd.Series({"a1": "Ann", "a2": "Ann", "a3": "Ann", "b1": "Ben", "b2": "Ben", "b3": "Ben"})
tiny_clusters = pd.Series({r: "one" for r in tiny_truth.index})
print(pd.DataFrame({"cluster": tiny_clusters, "true person": tiny_truth}))

**B-cubed** gives each record a score, then takes the mean over all records:

- **precision** of a record = the share of its cluster that is the same person as the record. For a1: 3 of the 6 records in its cluster are Ann, so 3 ÷ 6 = 0.5.
- **recall** of a record = the share of its person's records that are in its cluster. For a1: all 3 of Ann's records are in its cluster, so 3 ÷ 3 = 1.0.

> **Predict.** What are the B-cubed precision and recall of the tiny example?

## 2. Your turn: B-cubed

Complete `bcubed`. `both` is already there: for each record, the number of records with the same cluster **and** the same entity.

In [ ]:
def bcubed(clusters, entity):
    """B-cubed precision and recall: one score per record, then the mean over the records."""
    df = pd.DataFrame({"cluster": clusters, "entity": entity})
    both = df.groupby(["cluster", "entity"]).cluster.transform("size")    # records with the same cluster and entity
    c_size = df.groupby("cluster").cluster.transform("size")              # records in the record's cluster
    e_size = df.groupby("entity").entity.transform("size")                # records of the record's entity
    return round(float((both / c_size).mean()), 3), round(float((both / e_size).mean()), 3)

tiny_b3 = bcubed(tiny_clusters, tiny_truth)
print("B-cubed precision and recall:", tiny_b3)

In [ ]:
expect_hash('tiny_b3', tiny_b3, 'd75b198347fc2374')

## 3. Measure Larkfield's clusters

Run the next cell. It measures three things on the test split:

- the **matcher's pairs**: the pairs that the rules called the same (Module 2's pair measures);
- the **clusters of connected components**: every pair of records inside one cluster counts as "called the same", also pairs that no rule ever called the same;
- the **clusters of safe clustering**.

In [ ]:
t = truth().set_index("record_id")
test_ids = t.index[t.split == "test"]

def measures(cl):
    """B-cubed on the test-split records (clusters made from all records), cluster pairs, exact clusters."""
    df = pd.DataFrame({"cluster": cl, "entity": t.entity_id.reindex(cl.index)})
    df = df.assign(c_size=df.groupby("cluster").cluster.transform("size"),
                   both=df.groupby(["cluster", "entity"]).cluster.transform("size"),
                   e_size=df.groupby("entity").entity.transform("size"))
    test = df.loc[test_ids]
    b3p, b3r = round(float((test.both / test.c_size).mean()), 3), round(float((test.both / test.e_size).mean()), 3)
    members = df.rename_axis("record_id").reset_index()[["record_id", "cluster"]]
    cp = members.merge(members, on="cluster", suffixes=("_l", "_r"))
    cp = cp[cp.record_id_l < cp.record_id_r]
    pm = pair_measures(cp)
    sizes = cl.value_counts()
    return {"entities": cl.nunique(), "largest": int(sizes.max()), "B3 precision": b3p, "B3 recall": b3r,
            "cluster-pair precision": pm["precision"], "cluster-pair recall": pm["recall"]}

table = pd.DataFrame({"matcher's pairs": {**pair_measures(rules)}, "connected components": measures(cc),
                      "safe clustering": measures(safe)}).T
print(table[["entities", "largest", "precision", "recall", "B3 precision", "B3 recall", "cluster-pair precision",
             "cluster-pair recall"]].to_string())

In [ ]:
expect('B-cubed precision of connected components', table.loc['connected components', 'B3 precision'], 0.917, tolerance=0.001)

In [ ]:
expect('B-cubed precision of safe clustering', table.loc['safe clustering', 'B3 precision'], 0.976, tolerance=0.001)

> **Check.** The rules' pairs have precision 0.723. The pairs inside the connected components have precision 0.524: lower, because clustering adds pairs that no rule called the same. B-cubed precision goes from 0.917 to 0.976 with safe clustering.

## 4. Where the wrong pairs hide

A cluster of n records holds n × (n − 1) ÷ 2 pairs. A cluster of 14 records holds 91 pairs. Run the next cell: it counts the pairs inside the clusters by cluster size (8 means 8 or more), and the wrong ones.

In [ ]:
# Where are the wrong pairs inside the connected components? Count them per cluster size.
members = cc.rename("cluster").rename_axis("record_id").reset_index()
cp = members.merge(members, on="cluster", suffixes=("_l", "_r"))
cp = cp[cp.record_id_l < cp.record_id_r]
ent = t.entity_id
cp = cp.assign(wrong=cp.record_id_l.map(ent) != cp.record_id_r.map(ent), size=cp.cluster.map(cc.value_counts()))
by_size = cp.groupby(cp["size"].clip(upper=8)).agg(pairs=("wrong", "size"), wrong=("wrong", "sum"))
by_size["clusters"] = cc.value_counts().clip(upper=8).value_counts().reindex(by_size.index)
print(by_size)
big_share = round(float(cp[cp["size"] > 5].wrong.sum() / cp.wrong.sum()), 3)
print(f"clusters of more than 5 records hold {big_share:.1%} of the wrong pairs inside clusters")

In [ ]:
expect('share of wrong pairs in clusters over 5', big_share, 0.488, tolerance=0.001)

> **Check.** The few clusters of more than 5 records hold 48.8% of the wrong pairs. The pair measures of the matcher never show them, because the matcher never called those pairs the same.

The project prints the same measures for any run:

```sh
match eval --run rules
match eval --run lgbm
```

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [New records every day](https://learning.nextia-ai.com/courses/record-matching/m08/new-records-every-day/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/record-matching/m07/cluster-measures/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every customer record and every product listing here is synthetic: our code made them from invented people and products. No real person is in this data.